# Baseline Performance Notebook

- This notebook is used to infer the baseline performance of the model.
- The model used is llava-1.5-7b-hf
- The dataset used is abhay2812/vqa_rad_full
- The model is loaded from huggingface and the weights are downloaded to the local directory
- The model is then used to infer the answers for the questions in the dataset
- The answers are then compared to the ground truth answers to calculate the accuracy of the model


## Download the model and dataset

In [1]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="llava-hf/llava-1.5-7b-hf",
    local_dir="../models/llava-1.5-7b-hf"
)

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

'/ix/cs2770_2026s/abn80/cs2770_project/models/llava-1.5-7b-hf'

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="abhay2812/vqa-rad",
    repo_type="dataset",
    local_dir="../data/vqa-rad"
)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

'/ix/cs2770_2026s/abn80/cs2770_project/data/vqa-rad'

## Test model on one example

In [4]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
t = torch.tensor([1.0]).cuda()
print(t)

2.11.0+cu128
True
NVIDIA RTX PRO 6000 Blackwell Server Edition
tensor([1.], device='cuda:0')


In [11]:
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration

del model
torch.cuda.empty_cache()

model = LlavaForConditionalGeneration.from_pretrained(
    "../models/llava-1.5-7b-hf",
    torch_dtype=torch.float16,
).to("cuda")

print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB")
processor = AutoProcessor.from_pretrained(model_path)

print(f"Device: {model.device}")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

GPU memory: 15.9 GB
Device: cuda:0
GPU memory used: 15.9 GB


In [12]:
# Run the single example inference first, then check memory
from PIL import Image
from datasets import load_from_disk, load_dataset

ds = load_dataset("abhay2812/vqa-rad", cache_dir="../data/vqa-rad-cache")
sample = ds['test'][0]
print(sample)
img = sample['image'].convert("RGB")
prompt = f"USER: <image>\n{sample['question']}\nASSISTANT:"
inputs = processor(text=prompt, images=img, return_tensors="pt").to(model.device, torch.float16)

with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=100, do_sample=False)

pred = processor.decode(output[0], skip_special_tokens=True).split("ASSISTANT:")[-1].strip()

print(f"Question:  {sample['question']}")
print(f"GT Answer: {sample['answer']}")
print(f"Predicted: {pred}")
print(f"Type: {sample['question_type_primary']} | {sample['answer_type']}")
print(f"\nGPU memory now: {torch.cuda.memory_allocated()/1e9:.1f} GB")

{'qid': 10, 'image_name': 'synpic42202.jpg', 'image_organ': 'CHEST', 'question': 'Is there evidence of an aortic aneurysm?', 'answer': 'yes', 'answer_normalized': 'yes', 'answer_type': 'CLOSED', 'question_type_primary': 'PRES', 'question_type_raw': 'PRES', 'phrase_type': 'test_freeform', 'evaluation': 'not evaluated', 'split': 'test', 'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1024x1291 at 0x7FAB9427DD90>}
Question:  Is there evidence of an aortic aneurysm?
GT Answer: yes
Predicted: The image shows a close-up of a person's chest, which includes a heart and a pacemaker. However, there is no clear evidence of an aortic aneurysm in the image. Aortic aneurysms are typically characterized by a bulging or ballooning of the aorta, which is the main artery that carries blood from the heart to the rest of the body. The presence of an aortic aneurys
Type: PRES | CLOSED

GPU memory now: 15.9 GB


In [13]:
# Cell - Define prompts
def get_prompt(question, answer_type):
    if answer_type == 'CLOSED':
        return f"USER: <image>\n{question} Answer with only yes or no.\nASSISTANT:"
    else:
        return f"USER: <image>\n{question} Answer in a few words.\nASSISTANT:"

# Test both
sample_closed = ds['test'].filter(lambda x: x['answer_type'] == 'CLOSED')[0]
sample_open = ds['test'].filter(lambda x: x['answer_type'] == 'OPEN')[0]

for sample, label in [(sample_closed, 'CLOSED'), (sample_open, 'OPEN')]:
    img = sample['image'].convert("RGB")
    prompt = get_prompt(sample['question'], sample['answer_type'])
    inputs = processor(text=prompt, images=img, return_tensors="pt").to("cuda", torch.float16)
    
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=50, do_sample=False)
    
    pred = processor.decode(output[0], skip_special_tokens=True).split("ASSISTANT:")[-1].strip()
    
    print(f"\n--- {label} ---")
    print(f"Question:  {sample['question']}")
    print(f"GT Answer: {sample['answer']}")
    print(f"Predicted: {pred}")

Filter:   0%|          | 0/450 [00:00<?, ? examples/s]

Filter:   0%|          | 0/450 [00:00<?, ? examples/s]


--- CLOSED ---
Question:  Is there evidence of an aortic aneurysm?
GT Answer: yes
Predicted: No

--- OPEN ---
Question:  How is the patient oriented?
GT Answer: Posterior-Anterior
Predicted: Left
